# ML-03 — Frame Your Lane as an ML Task

This notebook translates the **Content Refresh Priority** question into a supervised binary classification and probability-ranking task.

## 1. My lane as an ML task (type)

My chosen lane is **Content Refresh Priority Prediction**, framed as **supervised binary classification with calibrated probability ranking**.

- **Binary Classification:** Predict whether a content item is a priority refresh candidate (`1`) vs. lower priority (`0`).
- **Probability Ranking:** Rank candidate pages by predicted probability `P(y = 1 | x)` so editorial teams can review the top *K* pages (e.g., Top 10, Top 20, Top 50) each week.

## 2. Target or proxy

- **In the Week-5/6 practice cohort (`w05_ml_practice_dataset.csv`):** The binary target column is `target` (`1` = priority refresh candidate, `0` = monitor).
- **In the 30,000-row FlyRank starter cohort (`content_refresh_anonymized.csv`):** The binary target proxy is `is_declining_label` (`1` when `trend_direction == 'down'`, else `0`).
- **Operational Action Tiers:** Scores and probabilities map to three human-readable action buckets: `REVIEW_NOW`, `PRIORITIZE`, and `MONITOR`.

## 3. Success metric

- **Primary Classification Metrics:** **F1-score** and **ROC AUC**.
- **Primary Queue Ranking Metrics:** **Precision@10**, **Precision@20**, and **Precision@50**.
- **Supporting Metrics:** Accuracy, Precision, Recall, and the confusion matrix (`TP`, `FP`, `TN`, `FN`) compared against both the positive base rate and the Week-4 hand-crafted rule baseline on the exact same test split.

## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One row = **one content item (`content_id` or `item_id`) evaluated at the end of the observation window**.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
df_starter = pd.read_csv(REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df_starter["is_declining_label"] = (df_starter["trend_direction"].astype(str).str.lower() == "down").astype(int)

print("FlyRank Starter Cohort Shape:", df_starter.shape)
print("One row per content_id? Unique content_ids:", df_starter["content_id"].nunique(), "==", len(df_starter))

preview_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update", "is_declining_label"
]
print("\nUnit of Analysis Preview (first 5 rows):")
print(df_starter[preview_cols].head().to_string(index=False))

w05_path = REPO_ROOT / "work" / "data" / "w05_ml_practice_dataset.csv"
if w05_path.exists():
    df_w05 = pd.read_csv(w05_path)
    print("\nW05 Practice Cohort Shape:", df_w05.shape)
    print(df_w05.head().to_string(index=False))


FlyRank Starter Cohort Shape: (30000, 45)
One row per content_id? Unique content_ids: 30000 == 30000

Unit of Analysis Preview (first 5 rows):
          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update  is_declining_label
content_304f48230142 client_f369cb89fc             3803          29 0.76          10.6                      20                   1
content_a1fb4e703a9e client_4e07408562            15320           7 0.05          20.3                      25                   1
content_9aa793d4d895 client_7f2253d7e2            12581          11 0.09          36.5                      20                   1
content_331d6c4de07b client_19581e27de            11751          58 0.49           6.2                      22                   0
content_d99b7a2d90ca client_3fdba35f04            19140          24 0.13          44.0                      14                   1

W05 Practice Cohort Shape: (100, 6)
 item_id  impressions  clicks  sta

## 5. Why ML beats a fixed rule here

A fixed threshold rule cannot capture continuous trade-offs across multiple SEO metrics. Standardized Logistic Regression learns continuous weights across impressions, clicks, staleness, and ranking position, allowing smoother prioritization and better tie-breaking than manually defined step cutoffs.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`